In [1]:
"""
Dermalytix — FastAPI Backend (Colab Version)
=============================================
Run all cells top to bottom in order.

Endpoints:
  POST /register
  POST /login
  POST /analyze
  GET  /history/{user_id}
  GET  /health
"""

# ══════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ══════════════════════════════════════════════════════════
# Run this cell first

import subprocess
subprocess.run([
    "pip", "install",
    "fastapi", "uvicorn", "python-multipart",
    "sqlalchemy", "passlib[bcrypt]", "python-jose[cryptography]",
    "pillow", "torch", "torchvision", "onnxruntime",
    "nest-asyncio", "pyngrok", "-q"
])
print("✅ Dependencies installed")

✅ Dependencies installed


In [2]:
# ══════════════════════════════════════════════════════════
# CELL 2 — Mount Drive + imports
# ══════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import os, io, uuid, random, time, shutil
from datetime import datetime, timedelta
from typing import Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image

from fastapi import FastAPI, HTTPException, Depends, UploadFile, File, Form, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel

from sqlalchemy import create_engine, Column, String, Float, Integer, DateTime, Boolean, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

from passlib.context import CryptContext
from jose import JWTError, jwt

import nest_asyncio
nest_asyncio.apply()

print("✅ Imports done")

Mounted at /content/drive
✅ Imports done


In [3]:
# ══════════════════════════════════════════════════════════
# CELL 3 — Config
# ══════════════════════════════════════════════════════════

# ── Paths ──
MODEL_PATH  = "/content/drive/MyDrive/DERMALYTIX_MODEL/dermalytix_best_v2.pth"
UPLOAD_DIR  = "/content/drive/MyDrive/DERMALYTIX_UPLOADS"
DB_PATH     = "/content/drive/MyDrive/DERMALYTIX_MODEL/dermalytix.db"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ── Auth ──
SECRET_KEY    = "dermalytix-secret-key-change-in-production"
ALGORITHM     = "HS256"
TOKEN_EXPIRE  = 60 * 24   # 24 hours in minutes

# ── Model ──
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7
CLASS_NAMES = [
    "Acne", "Dark circles", "dark spots",
    "dry", "normal", "oily", "wrinkles:FINE LINES"
]
CONFIDENCE_THRESHOLD = 0.50
TOP_N_CONDITIONS     = 3    # return top 3 predictions

print(f"✅ Config loaded")
print(f"   Device  : {DEVICE}")
print(f"   Model   : {MODEL_PATH}")

✅ Config loaded
   Device  : cpu
   Model   : /content/drive/MyDrive/DERMALYTIX_MODEL/dermalytix_best_v2.pth


In [4]:
# ══════════════════════════════════════════════════════════
# CELL 4 — Database setup
# ══════════════════════════════════════════════════════════

engine = create_engine(
    f"sqlite:///{DB_PATH}",
    connect_args={"check_same_thread": False}
)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# ── Models ──
class User(Base):
    __tablename__ = "users"
    id         = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    name       = Column(String)
    email      = Column(String, unique=True, index=True)
    password   = Column(String)
    age        = Column(Integer, nullable=True)
    gender     = Column(String, nullable=True)
    skin_type  = Column(String, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

class AnalysisSession(Base):
    __tablename__ = "analysis_sessions"
    id            = Column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    user_id       = Column(String, index=True)
    image_url     = Column(String)
    condition     = Column(String)
    display_label = Column(String)
    confidence    = Column(Float)
    severity      = Column(String)
    skin_type     = Column(String, nullable=True)
    routine       = Column(Text)   # JSON string
    see_doctor    = Column(Boolean, default=False)
    created_at    = Column(DateTime, default=datetime.utcnow)

Base.metadata.create_all(bind=engine)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

print("✅ Database ready →", DB_PATH)

/tmp/ipykernel_2094/459222640.py:10: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


✅ Database ready → /content/drive/MyDrive/DERMALYTIX_MODEL/dermalytix.db


In [5]:
# ══════════════════════════════════════════════════════════
# CELL 5 — Auth helpers
# ══════════════════════════════════════════════════════════

pwd_context   = CryptContext(schemes=["bcrypt"], deprecated="auto")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)

def create_token(data: dict) -> str:
    payload = data.copy()
    payload["exp"] = datetime.utcnow() + timedelta(minutes=TOKEN_EXPIRE)
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def get_current_user(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        email   = payload.get("sub")
        if not email:
            raise HTTPException(status_code=401, detail="Invalid token")
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid token")
    user = db.query(User).filter(User.email == email).first()
    if not user:
        raise HTTPException(status_code=401, detail="User not found")
    return user

print("✅ Auth helpers ready")

✅ Auth helpers ready


In [6]:
# ══════════════════════════════════════════════════════════
# CELL 6 — Load AI model
# ══════════════════════════════════════════════════════════

def build_model(num_classes=7):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )
    return model

checkpoint   = torch.load(MODEL_PATH, map_location=DEVICE)
skin_model   = build_model(NUM_CLASSES)
skin_model.load_state_dict(checkpoint["model_state_dict"])
skin_model    = skin_model.to(DEVICE)
skin_model.eval()

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print(f"✅ Model loaded → {NUM_CLASSES} classes")
print(f"   Classes: {CLASS_NAMES}")

✅ Model loaded → 7 classes
   Classes: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']


In [7]:
# ══════════════════════════════════════════════════════════
# CELL 7 — Label map
# ══════════════════════════════════════════════════════════

LABEL_MAP = {
    "Acne": {
        "display":     "Acne / Pimples / Breakouts",
        "description": "Active acne lesions including papules, pustules or comedones",
        "icon":        "🔴"
    },
    "Dark circles": {
        "display":     "Dark Circles / Periorbital Hyperpigmentation",
        "description": "Darkening of the skin under and around the eyes",
        "icon":        "👁️"
    },
    "dark spots": {
        "display":     "Dark Spots / Hyperpigmentation / Post-Acne Marks",
        "description": "Uneven skin tone, dark patches or marks left after acne or sun damage",
        "icon":        "🟤"
    },
    "dry": {
        "display":     "Dry Skin / Dehydrated Skin",
        "description": "Skin lacking moisture, may feel tight, flaky or rough",
        "icon":        "🏜️"
    },
    "normal": {
        "display":     "Normal / Balanced Skin",
        "description": "Well-balanced skin with no major concerns",
        "icon":        "✨"
    },
    "oily": {
        "display":     "Oily Skin / Seborrhoea",
        "description": "Excess sebum production causing shine and enlarged pores",
        "icon":        "💧"
    },
    "wrinkles:FINE LINES": {
        "display":     "Wrinkles / Fine Lines / Ageing Skin",
        "description": "Signs of ageing including fine lines, wrinkles and loss of elasticity",
        "icon":        "〰️"
    },
}

def get_display_label(raw_class):
    return LABEL_MAP.get(raw_class, {
        "display":     raw_class,
        "description": "",
        "icon":        "🔍"
    })

print("✅ Label map ready")

✅ Label map ready


In [8]:
# ══════════════════════════════════════════════════════════
# CELL 8 — Paste your FULL recommendation engine here
# ══════════════════════════════════════════════════════════
# Copy everything from 03_recommendation_engine.ipynb
# from ROUTINES = { ... } all the way to merge_routines()
# Paste it below this comment:

# ── PASTE RECOMMENDATION ENGINE CODE HERE ──

# (After pasting, the ROUTINES dict, get_severity(),
#  get_recommendation() and merge_routines() will be available)

print("✅ Recommendation engine loaded")

✅ Recommendation engine loaded


In [9]:
# ══════════════════════════════════════════════════════════
# CELL 9 — Prediction function
# ══════════════════════════════════════════════════════════

def predict_image(image_bytes: bytes, top_n: int = TOP_N_CONDITIONS):
    """
    Run model on image bytes.
    Returns top N predictions with confidence scores.
    """
    img        = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    tensor     = img_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = skin_model(tensor)
        probs  = F.softmax(output, dim=1)[0]

    # Get top N predictions
    top_probs, top_idxs = probs.topk(top_n)

    predictions = []
    for prob, idx in zip(top_probs, top_idxs):
        condition  = CLASS_NAMES[idx.item()]
        confidence = prob.item()
        if confidence >= CONFIDENCE_THRESHOLD:
            label = get_display_label(condition)
            predictions.append({
                "condition":   condition,
                "display":     label["display"],
                "description": label["description"],
                "icon":        label["icon"],
                "confidence":  round(confidence, 3),
            })

    return predictions

print("✅ Prediction function ready")

✅ Prediction function ready


In [10]:
# ══════════════════════════════════════════════════════════
# CELL 10 — Pydantic schemas
# ══════════════════════════════════════════════════════════

class RegisterRequest(BaseModel):
    name:      str
    email:     str
    password:  str
    age:       Optional[int]   = None
    gender:    Optional[str]   = None
    skin_type: Optional[str]   = None

class LoginResponse(BaseModel):
    access_token: str
    token_type:   str
    user_id:      str
    name:         str

class HistoryItem(BaseModel):
    session_id:    str
    condition:     str
    display_label: str
    confidence:    float
    severity:      str
    created_at:    str

print("✅ Schemas ready")

✅ Schemas ready


In [11]:
# ══════════════════════════════════════════════════════════
# CELL 11 — FastAPI app + routes
# ══════════════════════════════════════════════════════════

import json

app = FastAPI(
    title       = "Dermalytix API",
    description = "AI-powered skin analysis for Pakistani users",
    version     = "1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)

# ── Health check ──────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status":  "running",
        "model":   "EfficientNetB0",
        "classes": NUM_CLASSES,
        "version": "1.0.0"
    }

# ── Register ──────────────────────────────────────────────
@app.post("/register")
def register(req: RegisterRequest, db: Session = Depends(get_db)):
    existing = db.query(User).filter(User.email == req.email).first()
    if existing:
        raise HTTPException(status_code=400, detail="Email already registered")

    user = User(
        name      = req.name,
        email     = req.email,
        password  = hash_password(req.password),
        age       = req.age,
        gender    = req.gender,
        skin_type = req.skin_type,
    )
    db.add(user)
    db.commit()
    db.refresh(user)

    token = create_token({"sub": user.email})
    return {
        "message":      "Account created successfully",
        "access_token": token,
        "token_type":   "bearer",
        "user_id":      user.id,
        "name":         user.name,
    }

# ── Login ─────────────────────────────────────────────────
@app.post("/login")
def login(
    form: OAuth2PasswordRequestForm = Depends(),
    db:   Session = Depends(get_db)
):
    user = db.query(User).filter(User.email == form.username).first()
    if not user or not verify_password(form.password, user.password):
        raise HTTPException(status_code=401, detail="Invalid email or password")

    token = create_token({"sub": user.email})
    return {
        "access_token": token,
        "token_type":   "bearer",
        "user_id":      user.id,
        "name":         user.name,
    }

# ── Analyze ───────────────────────────────────────────────
@app.post("/analyze")
async def analyze(
    image:        UploadFile = File(...),
    name:         str        = Form(...),
    age:          int        = Form(...),
    gender:       str        = Form(...),
    skin_type:    str        = Form(...),
    main_concern: str        = Form(...),
    db:           Session    = Depends(get_db),
    current_user: User       = Depends(get_current_user),
):
    # 1. Validate image
    if image.content_type not in ["image/jpeg", "image/png", "image/webp"]:
        raise HTTPException(status_code=400, detail="Only JPEG, PNG, WebP images allowed")

    image_bytes = await image.read()
    if len(image_bytes) > 10 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="Image too large. Max 10MB")

    # 2. Save image
    filename  = f"{uuid.uuid4()}.jpg"
    save_path = os.path.join(UPLOAD_DIR, filename)
    with open(save_path, "wb") as f:
        f.write(image_bytes)
    image_url = f"/images/{filename}"

    # 3. Run model prediction
    predictions = predict_image(image_bytes, top_n=TOP_N_CONDITIONS)

    if not predictions:
        return {
            "session_id":    None,
            "is_uncertain":  True,
            "message":       "No condition detected with enough confidence. Please consult a dermatologist.",
            "predictions":   [],
            "routine":       None,
            "disclaimer":    "Not a medical diagnosis."
        }

    # 4. Run recommendation engine
    recommendation = merge_routines(
        conditions_with_confidence = [
            {"condition": p["condition"], "confidence": p["confidence"]}
            for p in predictions
        ],
        skin_type = skin_type
    )

    # 5. Primary condition (highest confidence)
    primary    = predictions[0]
    label_info = get_display_label(primary["condition"])

    # 6. Save to database
    session = AnalysisSession(
        user_id       = current_user.id,
        image_url     = image_url,
        condition     = primary["condition"],
        display_label = label_info["display"],
        confidence    = primary["confidence"],
        severity      = recommendation.get("conditions", [{}])[0].get("severity", "mild"),
        skin_type     = skin_type,
        routine       = json.dumps(recommendation.get("routine")),
        see_doctor    = recommendation.get("routine", {}).get("see_doctor", False),
    )
    db.add(session)
    db.commit()
    db.refresh(session)

    # 7. Return full response
    return {
        "session_id": session.id,
        "user": {
            "name":         name,
            "age":          age,
            "gender":       gender,
            "skin_type":    skin_type,
            "main_concern": main_concern,
        },
        "predictions":  predictions,
        "primary": {
            "condition":    primary["condition"],
            "display_label": label_info["display"],
            "description":  label_info["description"],
            "icon":         label_info["icon"],
            "confidence":   primary["confidence"],
            "severity":     session.severity,
        },
        "routine":        recommendation.get("routine"),
        "is_uncertain":   recommendation.get("is_uncertain", False),
        "saved_to_history": True,
        "disclaimer":     "⚠️ Not a medical diagnosis. Results are AI-generated suggestions only. Please consult a dermatologist for professional advice.",
    }

# ── History ───────────────────────────────────────────────
@app.get("/history/{user_id}")
def get_history(
    user_id:      str,
    db:           Session = Depends(get_db),
    current_user: User    = Depends(get_current_user),
):
    # Only allow users to see their own history
    if current_user.id != user_id:
        raise HTTPException(status_code=403, detail="Access denied")

    sessions = (
        db.query(AnalysisSession)
        .filter(AnalysisSession.user_id == user_id)
        .order_by(AnalysisSession.created_at.desc())
        .all()
    )

    return {
        "user_id":  user_id,
        "total":    len(sessions),
        "history": [
            {
                "session_id":    s.id,
                "condition":     s.condition,
                "display_label": s.display_label,
                "confidence":    s.confidence,
                "severity":      s.severity,
                "skin_type":     s.skin_type,
                "see_doctor":    s.see_doctor,
                "image_url":     s.image_url,
                "created_at":    s.created_at.strftime("%Y-%m-%d %H:%M"),
            }
            for s in sessions
        ]
    }

print("✅ All routes registered")
print("\nEndpoints:")
print("  GET  /health")
print("  POST /register")
print("  POST /login")
print("  POST /analyze")
print("  GET  /history/{user_id}")

✅ All routes registered

Endpoints:
  GET  /health
  POST /register
  POST /login
  POST /analyze
  GET  /history/{user_id}


In [12]:
# Add your ngrok authtoken
from pyngrok import ngrok

ngrok.set_auth_token("3EAR4zwLjiCwBT1SIpJ8oLzUVxO_7wCcWyqY2DsjeXjjkCPF7")  # ← paste your token
print("✅ ngrok authenticated")

✅ ngrok authenticated


In [13]:
# ══════════════════════════════════════════════════════════
# CELL 12 — Run server + get public URL
# ══════════════════════════════════════════════════════════

import uvicorn
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("\n" + "="*55)
print("  DERMALYTIX API IS LIVE!")
print("="*55)
print(f"\n  🌐 Public URL : {public_url}")
print(f"  📚 API Docs  : {public_url}/docs")
print(f"  ❤️  Health    : {public_url}/health")
print("\n  Share the Public URL with Antigravity!")
print("="*55)

# Run server
import uvicorn
import asyncio

# Fix for Colab's event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()


  DERMALYTIX API IS LIVE!

  🌐 Public URL : NgrokTunnel: "https://sandy-predator-perpetual.ngrok-free.dev" -> "http://localhost:8000"
  📚 API Docs  : NgrokTunnel: "https://sandy-predator-perpetual.ngrok-free.dev" -> "http://localhost:8000"/docs
  ❤️  Health    : NgrokTunnel: "https://sandy-predator-perpetual.ngrok-free.dev" -> "http://localhost:8000"/health

  Share the Public URL with Antigravity!


INFO:     Started server process [2094]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     182.177.94.82:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     182.177.94.82:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     182.177.94.82:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     182.177.94.82:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     182.177.94.82:0 - "GET /health HTTP/1.1" 200 OK


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/passlib/handlers/bcrypt.py", line 620, in _load_backend_mixin
    version = _bcrypt.__about__.__version__
              ^^^^^^^^^^^^^^^^^
AttributeError: module 'bcrypt' has no attribute '__about__'


INFO:     182.177.94.82:0 - "POST /register HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error